<a href="https://colab.research.google.com/github/dhruvbit/Machine-project/blob/main/AI_dispatcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 18.3 MB/s eta 0:00:00


In [2]:
from pprint import pprint
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import transformers
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim


from transformers import AutoModel , AutoModelForSequenceClassification,AutoTokenizer
from tqdm.auto import tqdm
from torch.utils.data import random_split
import torchmetrics
import os

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device :{device}")
!pip install -q datasets

Using device :cuda


How the Dataset Was Augmented
To build a truly robust dispatcher, you can't just rely on the original data. That's why this dataset has been augmented using a professional technique called back-translation. This process created new training examples that are grammatically different but semantically similar to the original ones.

In [4]:
import pandas as pd

# Download official jsonl from Hugging Face repository
!curl -L -o databricks-dolly-15k.jsonl "https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl"

df = pd.read_json("databricks-dolly-15k.jsonl", lines=True)

print(df['category'].unique().tolist())
print("Total rows:", len(df))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   942  100   942    0     0   4143      0 --:--:-- --:--:-- --:--:--  4149
100 12.4M  100 12.4M    0     0  15.2M      0 --:--:-- --:--:-- --:--:--  151M
['closed_qa', 'classification', 'open_qa', 'information_extraction', 'brainstorming', 'general_qa', 'summarization', 'creative_writing']
Total rows: 15011


In [5]:
df = df.dropna().reset_index(drop=True)

In [6]:
unique_categories = df['category'].unique().tolist()


In [7]:
print("Initial categories in the dataset:\n")
pprint(unique_categories)

Initial categories in the dataset:

['closed_qa',
 'classification',
 'open_qa',
 'information_extraction',
 'brainstorming',
 'general_qa',
 'summarization',
 'creative_writing']


In [8]:
df.head(10).T

,0,1,2,3,4,5,6,7,8,9
instruction,When did Virgin Australia start operating?,Which is a species of fish? Tope or Rope,Why can camels survive for long without water?,"Alice's parents have three daughters: Amy, Jes...",When was Tomoaki Komorida born?,If I have more pieces at the time of stalemate...,"Given a reference text about Lollapalooza, whe...",Who gave the UN the land in NY to build their HQ,Why mobile is bad for human,Who was John Moses Browning?
context,"Virgin Australia, the trading name of Virgin A...",,,,Komorida was born in Kumamoto Prefecture on Ju...,Stalemate is a situation in chess where the pl...,Lollapalooza /ˌlɒləpəˈluːzə/ (Lolla) is an ann...,,,"John Moses Browning (January 23, 1855 – Novemb..."
response,Virgin Australia commenced services on 31 Augu...,Tope,Camels use the fat in their humps to keep them...,The name of the third daughter is Alice,"Tomoaki Komorida was born on July 10,1981.",No. \nStalemate is a drawn position. It doesn'...,Lollapalooze is an annual musical festival hel...,John D Rockerfeller,We are always engaged one phone which is not g...,John Moses Browning is one of the most well-kn...
category,closed_qa,classification,open_qa,open_qa,closed_qa,information_extraction,closed_qa,open_qa,brainstorming,information_extraction


In [9]:
#create a mapping dictionary
#key:old name  , value:new name

category_map = {
    "general_qa":"q_and_a",
    "open_qa":"q_and_a",
    "closed_qa":"q_and_a",
    "information_extraction":"information_distillation",
    "summarization":"information_distillation"
}

df['category'] = df['category'].replace(category_map)

print(df['category'])

0                         q_and_a
1                  classification
2                         q_and_a
3                         q_and_a
4                         q_and_a
                   ...           
15006               brainstorming
15007    information_distillation
15008                     q_and_a
15009                     q_and_a
15010                     q_and_a
Name: category, Length: 15011, dtype: object


In [10]:
#convert into model readable form

unique_categories = df['category'].unique()

cat2id = {category: i for i, category in enumerate(unique_categories)}

df['label'] = df['category'].map(cat2id)

print(df['label'])

id2cat = {id: category for category, id in cat2id.items()}

print(id2cat)

0        0
1        1
2        0
3        0
4        0
        ..
15006    3
15007    2
15008    0
15009    0
15010    0
Name: label, Length: 15011, dtype: int64
{0: 'q_and_a', 1: 'classification', 2: 'information_distillation', 3: 'brainstorming', 4: 'creative_writing'}


In [11]:
texts = df['instruction'].tolist()
labels = df['label'].tolist()


In [12]:
print(f"Total samples for classification: {len(texts)}\n")
print("Class distribution:")


for category, label_id in cat2id.items():
  count=labels.count(label_id)
  print(f" -label {label_id}: {category:<25} {count} samples")

Total samples for classification: 15011

Class distribution:
 -label 0: q_and_a                   7706 samples
 -label 1: classification            2136 samples
 -label 2: information_distillation  2694 samples
 -label 3: brainstorming             1766 samples
 -label 4: creative_writing          709 samples


In [13]:
num_samples = 10
random_state = 25

# Display a sample of instruction and label pairs.
display(df[['instruction', 'label']].sample(num_samples, random_state=random_state).style.hide(axis="index"))

instruction,label
Ways to use Pencil other than writing,3
What is sweep rowing?,2
How do you make a simple green salsa?,0
How do compilers use IR?,2
Do Ceanothus have shallow roots?,0
"GIven this paragraph about American film director David Fincher, what are his reported upcoming projects.",0
"Categorize the following ingredients as meat, cheese, or spread: mayonnaise, ham, swiss, muenster, turkey, mustard",1
Write a short poem about spring,4
Which national park is hardest to visit?,4
Why do human beings react slow on climate changes?,0


In [14]:
def download_bert(model_name="distilbert-base-uncased",local_path="/content/distilbert-base-uncased"):

  if os.path.isdir(local_path):
    print(f"base model'{model_name} already available at {local_path}")

  else:
    print(f"downloading the base model'{model_name} to {local_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    tokenizer.save_pretrained(local_path)
    model.save_pretrained(local_path)

    print("base model downlaoding and save successfully")

In [15]:
model_name="distilbert-base-uncased"
model_path="./distilbert-local-base"

# Ensure the model is downloaded
download_bert(model_name, model_path)

downloading the base model'distilbert-base-uncased to ./distilbert-local-base


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

base model downlaoding and save successfully


In [16]:
def load_bert(local_path="./distilbert-local-base", num_classes=2):
    """
    Loads a base transformer model from a local directory and adds a new
    sequence classification head.

    """

    print(
        f"Loading base model from {local_path} and adding a new head with {num_classes} classes.\n"
    )


    tokenizer = AutoTokenizer.from_pretrained(local_path)
    # Load the base pre-trained model and add a new, untrained sequence
    # classification head on top with the specified number of labels.
    model = AutoModelForSequenceClassification.from_pretrained(
        local_path, num_labels=num_classes
    )


    print("\nModel and tokenizer loaded successfully.")


    return model, tokenizer

In [17]:
num_classes = df['category'].nunique()
bert_model , bert_tokenizer = load_bert(model_path,
                                        num_classes=num_classes
                                        )

Loading base model from ./distilbert-local-base and adding a new head with 5 classes.



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: ./distilbert-local-base
Key                   | Status  | 
----------------------+---------+-
pre_classifier.weight | MISSING | 
classifier.weight     | MISSING | 
classifier.bias       | MISSING | 
pre_classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model and tokenizer loaded successfully.


In [18]:
class InstructionDataset(Dataset):
    """
    Custom PyTorch Dataset for text classification.

    This Dataset class stores raw texts and their corresponding labels. It is
    designed to work efficiently with a Hugging Face tokenizer, performing
    tokenization on the fly for each sample when it is requested.

    Args:
        texts (list[str]): A list of raw text strings.
        labels (list[int]): A list of integer labels corresponding to the texts.
        tokenizer: A Hugging Face tokenizer instance used for processing text.
    """
    def __init__(self, texts, labels, tokenizer):




        self.texts = texts

        self.labels = labels

        self.tokenizer = tokenizer



    def __len__(self):
        """Returns the total number of samples in the dataset."""
        return len(self.texts)

    def __getitem__(self, idx):
        """
        Retrieves and processes one sample from the dataset.

        For a given index, this method fetches the corresponding text and label,
        tokenizes the text, and returns a dictionary of tensors.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            dict: A dictionary containing the tokenized inputs ('input_ids',
                  'attention_mask') and the 'labels' as tensors.
        """




        text = self.texts[idx]
        label = self.labels[idx]


        encoding = self.tokenizer(text,truncation=True, max_length=512)


        encoding['labels'] = torch.tensor(label,dtype=torch.long)



        return encoding



In [19]:
verify_class = InstructionDataset(texts,labels,bert_tokenizer)
print(f"Dataset initialized with {len(verify_class)} total samples.")
print("-"*50)

print("isplaying the first 3 instructions and labels from your data:\n")
for i in range(3):
  print(f"Instruction: {verify_class.texts[i]}")
  print(f"label:       {verify_class.labels[i]}")

print("-"*50)


Dataset initialized with 15011 total samples.
--------------------------------------------------
isplaying the first 3 instructions and labels from your data:

Instruction: When did Virgin Australia start operating?
label:       0
Instruction: Which is a species of fish? Tope or Rope
label:       1
Instruction: Why can camels survive for long without water?
label:       0
--------------------------------------------------


In [20]:
full_dataset = InstructionDataset(texts, labels, bert_tokenizer)

In [21]:
def create_dataset_split(full_dataset, train_split_percentage=0.8):

    """
    Splits a full dataset into training and validation sets.

    Args:
        full_dataset: The complete PyTorch Dataset to be split.
        train_split_percentage: The percentage of the dataset to allocate
                                for the training set. Defaults to 0.8.

    Returns:
        A tuple containing the training dataset and validation dataset.
    """
    train_size = int(train_split_percentage * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = random_split(full_dataset,[train_size,val_size])

    return train_dataset, val_dataset


In [22]:
train_dataset,val_dataset = create_dataset_split(
    full_dataset,
    train_split_percentage=0.8
)

print(f"Training samples:    {len(train_dataset)}")
print(f"Validation samples:  {len(val_dataset)}")

Training samples:    12008
Validation samples:  3003


In [23]:

def create_data_collator(tokenizer):
    """
    Initializes and returns a data collator for dynamic padding.

    Args:
        tokenizer: A Hugging Face tokenizer instance.

    Returns:
        collator: A transformers.DataCollatorWithPadding instance.
    """

    ### START CODE HERE ###

    # Initialize the data collator
    collator = transformers.DataCollatorWithPadding(tokenizer)

    ### END CODE HERE ###

    return collator

In [24]:
sample_1 = bert_tokenizer("this is short ")
sample_1['label'] = torch.tensor(0)
sample_2 = bert_tokenizer("this particular sentence is quite a bit longer.")
sample_2['label'] = torch.tensor(1)
manual_batch = [sample_1, sample_2]

verify_function = create_data_collator(bert_tokenizer)
print("--- Before Collation (Original input_ids) ---")

for i, sample in enumerate(manual_batch):
  print(f"sample{i+1} 'input_ids': {sample['input_ids']}")
print("\n--- After Collation (Padded and batched) ---")
# Pass the list of samples to the collator and inspect the result
collated_batch = verify_function(manual_batch)

# Directly print the final tensor
print(collated_batch['input_ids'])

--- Before Collation (Original input_ids) ---
sample1 'input_ids': [101, 2023, 2003, 2460, 102]
sample2 'input_ids': [101, 2023, 3327, 6251, 2003, 3243, 1037, 2978, 2936, 1012, 102]

--- After Collation (Padded and batched) ---
tensor([[ 101, 2023, 2003, 2460,  102,    0,    0,    0,    0,    0,    0],
        [ 101, 2023, 3327, 6251, 2003, 3243, 1037, 2978, 2936, 1012,  102]])


In [25]:
data_collator = create_data_collator(bert_tokenizer)

In [26]:
#Construncting the data loader
def create_dataloader(train_dataset,val_dataset, batch_size ,collate_fn):
    """
    create and return DataLoader instance for training
    """

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=True,

    )

    #Create the DataLoader for Validate set
    val_loader  = DataLoader(
      val_dataset,
      batch_size = batch_size,
      collate_fn = collate_fn,
      shuffle = False
    )

    return train_loader, val_loader


In [27]:
batch_size = 16
verify_function = create_dataloader(train_dataset,val_dataset,batch_size,data_collator)

first_train_batch = next(iter(verify_function[0]))
print("shape of each tensor in the train batch")
for key, value in first_train_batch.items():
  print(f"-{key}: {value.shape}")

print()



first_val_batch = next(iter(verify_function[1]))
print("shape of each tensor in the val batch")
for key, value in first_val_batch.items():
  print(f"-{key}: {value.shape}")


shape of each tensor in the train batch
-input_ids: torch.Size([16, 27])
-token_type_ids: torch.Size([16, 27])
-attention_mask: torch.Size([16, 27])
-labels: torch.Size([16])

shape of each tensor in the val batch
-input_ids: torch.Size([16, 32])
-token_type_ids: torch.Size([16, 32])
-attention_mask: torch.Size([16, 32])
-labels: torch.Size([16])


In [28]:
batch_size = 32
train_loader, val_loader = create_dataloader(train_dataset, val_dataset, batch_size, data_collator)

In [29]:
# calculating the class_weight
def calculate_class_weights(train_dataset,device):
  """
    Calculates class weights for handling imbalanced datasets.

    Args:
        train_dataset (torch.utils.data.Subset):
            The training dataset, expected to be a subset object containing
            indices to the original dataset.
        device (torch.device):
            The device (e.g., 'cuda' or 'cpu') to place the final tensor on.

    Returns:
        torch.Tensor: A 1D tensor of class weights.
  """
  train_label_list = [train_dataset.dataset.labels[i] for i in train_dataset.indices]

  class_weights = compute_class_weight(
        class_weight = 'balanced',
        classes = np.unique(train_label_list),
        y = train_label_list
    )

  class_weights_tensor = torch.tensor(class_weights , dtype = torch.float)
  return class_weights_tensor.to(device)


In [30]:
verify_function  = calculate_class_weights(train_dataset,device)
print("Breakdown of weights per category")

sorted_categories = sorted(cat2id.items(),key=lambda item: item[1])
for category, label_id in sorted_categories:
  weight = verify_function[label_id].item()
  print(f"  - {category:<25}: {weight:.4f}")

Breakdown of weights per category
  - q_and_a                  : 0.3925
  - classification           : 1.3786
  - information_distillation : 1.1103
  - brainstorming            : 1.7045
  - creative_writing         : 4.1694


In [31]:
label = df['label'].unique()
print(label)

[0 1 2 3 4]


In [32]:
print(list(cat2id.keys()))

['q_and_a', 'classification', 'information_distillation', 'brainstorming', 'creative_writing']


In [33]:
class_weights = calculate_class_weights(train_dataset, device)

# Initialize the CrossEntropyLoss function with the calculated `class_weights`.
loss_function = nn.CrossEntropyLoss(weight=class_weights)

In [34]:
# Now onto your second optimization strategy. Fine-tuning a language model with millions
# of parameters is computationally expensive and time-consuming.
# A more pragmatic and often equally effective approach is
# partial fine-tuning (also known as parameter-efficient fine-tuning (PEFT)).

In [35]:
def partially_bert_layers(model,layer_to_train=3):
    """
    Freezes all but the last N transformer layers and the classification head
    of a DistilBERT model. The model is modified in-place.

    Args:
        model (transformers.DistilBertForSequenceClassification):
            The DistilBERT model to modify.

        layers_to_train (int):
            The number of final transformer layers to unfreeze for training.

    Returns:
        transformers.DistilBertForSequenceClassification:
            The same model instance, now modified with the specified layers
            unfrozen.
    """
    for param in model.parameters():
        param.requires_grad = False
    transformer_layer =model.distilbert.transformer.layer


    for i in range(layer_to_train):
      layer_to_unfreeze = transformer_layer[-(i + 1)]

      for param in layer_to_unfreeze.parameters():
          param.requires_grad = True


    for param in model.pre_classifier.parameters():
      param.requires_grad = True

    for param in model.classifier.parameters():
      param.requires_grad = True


    return model

In [36]:
verify_function  = partially_bert_layers(bert_model)

for name, param in verify_function.named_parameters():

  if 'layer.0.attention.q_lin.weight' in name:
    print(f"parameter from an early layer: {name}")
    print(f'trainable(required_grad): {param.requires_grad}')

  if 'layer.4.attention.q_lin.weight' in name:
        print(f"\nParameter from a LATE layer: '{name}'")
        print(f"  - Trainable (requires_grad): {param.requires_grad}")

    # Check a parameter from the final classification head
  if 'classifier.weight' in name:
        print(f"\nParameter from the CLASSIFIER head: '{name}'")
        print(f"  - Trainable (requires_grad): {param.requires_grad}")

parameter from an early layer: distilbert.transformer.layer.0.attention.q_lin.weight
trainable(required_grad): False

Parameter from a LATE layer: 'distilbert.transformer.layer.4.attention.q_lin.weight'
  - Trainable (requires_grad): True

Parameter from the CLASSIFIER head: 'pre_classifier.weight'
  - Trainable (requires_grad): True

Parameter from the CLASSIFIER head: 'classifier.weight'
  - Trainable (requires_grad): True


In [37]:
layers_to_train = 5

# Set the learning rate for the optimizer
learning_rate = 1e-5

# Set the total number of training epochs
num_epochs = 3

In [38]:
if layers_to_train < 0:
    print(f"'layers_to_train' was set to {layers_to_train}, which is not valid.\n")
    print("The number of transformer layers to train cannot be negative. Setting layers_to_train=0")
    layers_to_train = 0

# Check if the value exceeds the total number of layers in the model
elif layers_to_train > 6:
    print(f"'layers_to_train' was set to {layers_to_train}, but DistilBERT only has 6 transformer layers.\n")
    print(f"Capping at the maximum value. Setting layers_to_train = 6")
    layers_to_train = 6

In [39]:
partial_finetune_model = partially_bert_layers(bert_model, layers_to_train)

In [40]:
def training_loop(model,train_loader,val_loader,loss_function,learning_rate,num_epochs,device):
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    num_classes = model.config.num_labels

    #init the number  of classes from the models
    val_accuracy = torchmetrics.Accuracy(task="multiclass",num_classes=num_classes).to(device)
    val_f1 = torchmetrics.F1Score(task="multiclass",num_classes=num_classes,average='macro').to(device)
    val_cm = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=num_classes).to(device)


    #create the main progress bar that iter
    epoch_loop = tqdm(range(num_epochs), desc="Training progress")


    for epoch in epoch_loop:
        model.train()

        train_loss_epoch = 0

        train_inner_loop = tqdm(train_loader, desc=f"epoch {epoch+1}/{num_epochs} Training", leave=False)

        for batch in train_inner_loop:
          input_ids = batch['input_ids'].to(device)
          attention_mask = batch['attention_mask'].to(device)
          labels = batch['labels'].to(device)

          optimizer.zero_grad()
          outputs = model(input_ids=input_ids,attention_mask=attention_mask)
          logits = outputs.logits

          loss = loss_function(logits,labels)
          train_loss_epoch +=loss.item()
          loss.backward()
          optimizer.step()

          train_inner_loop.set_postfix(loss=loss.item())
    train_loss_epoch /=len(train_loader)

     #_______validation phase ________

    model.eval()
    val_loss_epoch = 0
    val_inner_loop = tqdm(val_loader,desc=f"Epoch {epoch+1}/{num_epochs} validation",leave=False)

    with torch.no_grad():



      for batch in val_inner_loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,attention_mask=attention_mask)
        logits = outputs.logits

        val_loss = loss_function(logits,labels)
        logits = outputs.logits
        val_loss_epoch += val_loss.item()

        preds = torch.argmax(logits,dim=1)
        val_accuracy.update(preds,labels)
        val_f1.update(preds,labels)
        val_cm.update(preds,labels)

    val_loss_epoch /= len(val_loader)
    epoch_acc = val_accuracy.compute()
    epoch_f1 = val_f1.compute()

    val_accuracy.reset()
    val_f1.reset()

    epoch_loop.set_postfix(
       train_loss=f"{train_loss_epoch:.4f}",
       val_loss=f"{val_loss_epoch:.4f}",
       val_acc=f"{epoch_acc:.4f}"
   )

    tqdm.write(f"Epoch {epoch+1} Metrics -> val_loss: {val_loss_epoch:.4f},val_acc {epoch_acc:.4f}, val_F!: {epoch_f1:.4f}")

    print("training complete")

    final_confusion_matrix = val_cm.compute()
    final_results = {
        "val_loss":val_loss_epoch,
        "val_accuracy":epoch_acc.item(),
        "val_f1":epoch_f1.item(),
        "confusion_matrix":final_confusion_matrix.cpu()
    }

    return model, final_results


In [42]:
trained_finetuned_bert,partial_results = training_loop(
    model=partial_finetune_model,
    train_loader = train_loader,
    val_loader = val_loader,
    loss_function = loss_function,
    learning_rate = learning_rate,
    num_epochs=num_epochs,
    device=device
)

print("final validation metrices")
print(f"\nLoss:       {partial_results['val_loss']:.4f}")
print(f"Accuracy:   {partial_results['val_accuracy']:.4f}")
print(f"F1:         {partial_results['val_f1']:.4f}\n")

Training progress:   0%|          | 0/3 [00:00<?, ?it/s]

epoch 1/3 Training:   0%|          | 0/376 [00:00<?, ?it/s]

epoch 2/3 Training:   0%|          | 0/376 [00:00<?, ?it/s]

epoch 3/3 Training:   0%|          | 0/376 [00:00<?, ?it/s]

Epoch 3/3 validation:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 3 Metrics -> val_loss: 0.7288,val_acc 0.7329, val_F!: 0.7120
training complete
final validation metrices

Loss:       0.7288
Accuracy:   0.7329
F1:         0.7120

